In [1]:
!uv pip install -qqq \
    "torch>=2.8.0" "triton>=3.4.0" torchvision bitsandbytes \
    unsloth "unsloth_zoo>=2026.4.6" transformers==5.5.0 torchcodec timm

!uv pip install -q pymupdf pandas tqdm
!uv pip install -q requests

In [2]:
import re
import time
import json
import hashlib
import requests
from pathlib import Path
from tqdm.auto import tqdm
from typing import Optional, List
from pydantic import BaseModel, Field

import fitz  # PyMuPDF
import pandas as pd

from unsloth import FastModel
import torch

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
gemma4_models = [
    # Gemma-4 instruct models:
    "unsloth/gemma-4-E2B-it",
    "unsloth/gemma-4-E4B-it",
    "unsloth/gemma-4-31B-it",
    "unsloth/gemma-4-26B-A4B-it",
]

In [4]:
!pip -q install modelscope
import os; os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'
os.environ['HF_TOKEN'] = ''

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 42.2 MB/s eta 0:00:0000:0100:01


In [5]:
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-31B-it",
    dtype = None, # None for auto detection
    max_seq_length = 8192, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    token = os.getenv('HF_TOKEN'), # HF Token for gated models
    device_map = "balanced", # Use 2x Tesla T4s on Kaggle
)
inference = FastModel.for_inference(model)

==((====))==  Unsloth 2026.5.4: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1188 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

## Parse PDFs with data center specs
Extracts required information: total sq_m area, building materials, address.
Uses the full address to prepare a request body for OpenStreetMap API - to find latitude/longitude and the building polygon.

In [6]:
class DataCenterSpecsExtraction(BaseModel):
    data_center_name: Optional[str] = None
    address: Optional[str] = None
    colocation_space_sqm: Optional[float] = None
    building_materials: List[str] = Field(default_factory=list)
    maximum_power_load_kw: Optional[float] = None
    standby_power_total_kw: Optional[float] = None
    cooling_configuration: Optional[str] = None
    notes: List[str] = Field(default_factory=list)

class DataCenterLocation(BaseModel):
    latitude: Optional[str | float] = None
    longitude: Optional[str | float] = None

class ThermalCapacity(BaseModel):
    material: Optional[str] = None
    matched_material: Optional[str] = None
    phase: Optional[str] = None
    specific_heat_kj_per_kg_k: Optional[float] = None

class MaterialThermalCapacityExtraction(BaseModel):
    thermal_capacities: List[ThermalCapacity] = Field(default_factory=list)

In [7]:
class ProcessPDF:
    # def __init__(self, pdf_list: list[str]):
    #     self.pdf_list = pdf_list

    def extract_text(self, pdf_path, max_pages=99):
        pdf_path = Path(pdf_path)
        pages = []
        with fitz.open(pdf_path) as doc:
            for page_idx in range(min(len(doc), max_pages)):
                page = doc[page_idx]
                text = page.get_text("text")
                if text and text.strip():
                    pages.append(f"\n--- PAGE {page_idx + 1} ---\n{text}")
        return "\n".join(pages).strip()

    def chunk_text(self, text, chunk_size=9000, overlap=800):
        text = re.sub(r"\n{3,}", "\n\n", text).strip()
        if len(text) <= chunk_size:
            return [text]
        chunks = []
        start = 0
        while start < len(text):
            end = start + chunk_size
            chunks.append(text[start:end])
            start = max(end - overlap, end)
        return chunks

class ResponseParser:
    def json_from_model_output(self, output, original=None):
        print([output])
        # print([eval(output)])
        output = output.strip()
        try:
            return json.loads(output)
        except Exception:
            pass
        output = re.sub(r"^```(?:json)?", "", output, flags=re.IGNORECASE).strip()
        output = re.sub(r"```$", "", output).strip()
        try:
            return json.loads(output)
        except Exception:
            pass
        # Try extracting first JSON object
        match = re.search(r"\{.*\}", output, flags=re.DOTALL)
        if match:
            try:
                return json.loads(match.group(0))
            except Exception:
                pass
        return original

    def choose_best_value(self, values):
        """
        Simple merge strategy across chunks:
        pick the longest non-empty value, usually the most complete address/name.
        """
        clean_values = []
        for value in values:
            if value is None:
                continue
            value = str(value).strip()
            if value and value.lower() not in {"null", "none", "n/a", "unknown"}:
                clean_values.append(value)
        if not clean_values:
            return None
        unique_values = list(dict.fromkeys(clean_values))
        return max(unique_values, key=len)

    def create_nominatim_params(self, parsed, query_params):
        cleaned = {}
        for key in query_params:
            value = parsed.get(key)
            if value is None:
                continue
            value = str(value).strip()
            if not value or value.lower() in {"null", "none", "unknown", "n/a"}:
                continue
            cleaned[key] = value
        return cleaned

In [8]:
class PromptBuilder:
    def extract_data_center_specs(self, text):
        return f"""
    You are an information extraction system.
    
    Extract the following fields from the PDF text:
    
    - data_center_name
    - address
    - colocation_space_sqm
    - building_materials
    - maximum_power_load_kw
    - standby_power_total_kw
    - cooling_configuration
    - notes
    
    Return ONLY valid JSON.
    Do not include markdown.
    Do not explain your answer.
    Do not guess. Use null when a field is not found.
    
    Rules:
    - data_center_name should be the name of the data center, facility, site, campus, or colocation location.
    - Address should be the full physical address if available.
    - If multiple addresses exist, choose the one most directly associated with the data center.
    - Preserve the address as a single string.
    - Return colocation space in square meters.
    - Normalize materials to simple distinct names when possible.
    - Only set maximum_power_load_kw if the document explicitly states a maximum or total power load in kW.
    - If the document lists standby/generator power ratings, sum them into standby_power_total_kw.
    - Keep cooling_configuration close to the document wording.
    - Do not invent values.
    
    Expected JSON schema:
    {{
     "data_center_name": string|null,
     "address": string|null,
     "colocation_space_sqm": number|null,
     "building_materials": [string],
     "maximum_power_load_kw": number|null,
     "standby_power_total_kw": number|null,
     "cooling_configuration": string|null,
     "notes": [string]
    }}
    
    PDF text:
    \"\"\"
    {text}
    \"\"\"
    """.strip()

    def nominatim_param(self, extracted_record):
        return f"""
    You are preparing structured search parameters for the Nominatim geocoding API.
    
    Given this extracted data center record, return ONLY valid JSON using this schema:
    
    {{
      "amenity": string or null,
      "street": string or null,
      "city": string or null,
      "county": string or null,
      "state": string or null,
      "country": string or null,
      "postalcode": string or null,
      "countrycodes": string or null
    }}
    
    Rules:
    - Use structured search parameters only.
    - Do NOT return q.
    - Do NOT invent values.
    - Use null when a component is unknown.
    - street should be only building number plus street name, for example "151 Front Street West".
    - city should be the city/locality.
    - state should be the province/state/region.
    - country should be the country name.
    - postalcode should be only the postal code.
    - countrycodes should be ISO 3166-1 alpha-2 lowercase, for example "ca" for Canada.
    - amenity may contain the data center or facility name, but do not include floor numbers or marketing symbols.
    - Remove symbols like ® unless they are needed for the place name.
    - If the address contains floor numbers, suite numbers, building notes, or facility specs, do not put those in street.
    
    Extracted record:
    {json.dumps(extracted_record, ensure_ascii=False, indent=2)}
    """.strip()

    def extract_material_thermal_capacity(self, text, materials):
        return f"""
    You are an information extraction system.

    Extract the isobaric mass heat capacity Cp for each requested material from the PDF text.

    Requested materials:
    {json.dumps(materials, ensure_ascii=False)}

    Return ONLY valid JSON.
    Do not include markdown.
    Do not explain your answer.
    Do not guess. Use null when a material is not found.

    Rules:
    - Return one result per requested material, in the same order.
    - Preserve the requested material name in the material field.
    - Use the table column named isobaric mass heat capacity, Cp.
    - The table unit J g^-1 K^-1 is numerically equal to kJ kg^-1 K^-1.
    - If the exact material is not listed, use the closest unambiguous substance name only when it is clearly the same material.
    - Put the table substance name in matched_material.
    - Put the table phase in phase when available.
    - Do not invent values.

    Expected JSON schema:
    {{
      "thermal_capacities": [
        {{
          "material": string,
          "matched_material": string|null,
          "phase": string|null,
          "specific_heat_kj_per_kg_k": number|null
        }}
      ]
    }}

    PDF text:
    \"\"\"
    {text}
    \"\"\"
    """.strip()


class LLMFlow:
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer

    def model_inference(self, prompt, max_new_tokens):
        messages = [{
            "role": "user",
            "content": [{
                "type": "text",
                "text": prompt,
            }]
        }]
        
        inputs = self.tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_tensors="pt",
            tokenize=True,
            return_dict=True,
        ).to("cuda")
        
        self.model.eval()
        
        with torch.inference_mode():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                use_cache=True,
                temperature=0.0,
                do_sample=False,
            )
        
        input_len = inputs["input_ids"].shape[-1]
        generated_tokens = outputs[0][input_len:]
        return self.tokenizer.decode(generated_tokens, skip_special_tokens=True)

    def model_inference_image(self, prompt, images, max_new_tokens):
        messages = [{
            "role": "user",
            "content": [{
                "type": "text",
                "text": prompt,
            }]
        }]
        if images is not None:
            if not isinstance(images, list):
                images = [images]
            for img in images:
                messages[0]["content"].append({
                    "type": "image",
                    "image": img
                })
        inputs = self.tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_tensors="pt",
            tokenize=True,
            return_dict=True,
        ).to("cuda")
        
        self.model.eval()
        
        with torch.inference_mode():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                use_cache=True,
                temperature=0.0,
                do_sample=False,
            )
        
        input_len = inputs["input_ids"].shape[-1]
        generated_tokens = outputs[0][input_len:]
        return self.tokenizer.decode(generated_tokens, skip_special_tokens=True)
        
    def extract_fields_from_text(self, prompt, max_new_tokens=256):
        return self.model_inference(prompt, max_new_tokens)

    def extract_params_from_address_line(self, prompt, max_new_tokens=256):
        return self.model_inference(prompt, max_new_tokens)

    def create_nominatim_params0(self, parsed, query_params, max_new_tokens=256):
        
        # print(decoded)
        # parsed = json.loads(decoded)
        cleaned = {}
        for key in query_params:
            value = parsed.get(key)
            if value is None:
                continue
            value = str(value).strip()
            if not value or value.lower() in {"null", "none", "unknown", "n/a"}:
                continue
            cleaned[key] = value
        return cleaned

In [9]:
class LocationGrounding:
    NOMINATIM_ENDPOINT = "https://nominatim.openstreetmap.org/search"
    
    NOMINATIM_USER_AGENT = "DataCenterPDFGeocoder/0.1 (katerinaZakharova2000@gmail.com)"

    NOMINATIM_CACHE_PATH = Path("/nominatim_cache.json")
    
    ALLOWED_NOMINATIM_QUERY_PARAMS = {
        "amenity",
        "street",
        "city",
        "county",
        "state",
        "country",
        "postalcode",
        "countrycodes",
    }
    def load_nominatim_cache(self):
        if self.NOMINATIM_CACHE_PATH.exists():
            with open(self.NOMINATIM_CACHE_PATH, "r", encoding="utf-8") as f:
                return json.load(f)
        return {}
    
    
    def save_nominatim_cache(self, cache):
        with open(self.NOMINATIM_CACHE_PATH, "w", encoding="utf-8") as f:
            json.dump(cache, f, ensure_ascii=False, indent=2)

    def make_cache_key(self, params):
        canonical = json.dumps(params, ensure_ascii=False, sort_keys=True)
        return hashlib.sha256(canonical.encode("utf-8")).hexdigest()
    
    def nominatim_search_structured(
        self,
        structured_params,
        limit=3,
        addressdetails=1,
        extratags=1,
        namedetails=1,
        cache=None,
        min_delay_seconds=2.1,
    ):
        if cache is None:
            cache = self.load_nominatim_cache()
        params = dict(structured_params)
        params.update({
            "format": "jsonv2",
        #     "limit": int(limit),
        #     "addressdetails": int(addressdetails),
        #     "extratags": int(extratags),
        #     "namedetails": int(namedetails),
        })
        params.pop("q", None)
        cache_key = self.make_cache_key(params)
        if cache_key in cache:
            return cache[cache_key]
        headers = {
            "User-Agent": self.NOMINATIM_USER_AGENT,
            "Accept": "application/json",
        }
        time.sleep(min_delay_seconds)
        response = requests.get(
            self.NOMINATIM_ENDPOINT,
            params=params,
            headers=headers,
            timeout=30,
        )
        response.raise_for_status()
        data = response.json()
        cache[cache_key] = {"request_params": params, "results": data}
        self.save_nominatim_cache(cache)
        return cache[cache_key]

    def normalize_nominatim_result(self, result):
        return {
            "latitude": result.get("lat"),
            "longitude": result.get("lon"),
            "geocoded_display_name": result.get("display_name"),
            "geocoded_class": result.get("class"),
            "geocoded_type": result.get("type"),
            "geocoded_importance": result.get("importance"),
            "geocoded_osm_type": result.get("osm_type"),
            "geocoded_osm_id": result.get("osm_id"),
            "geocoded_place_id": result.get("place_id"),
            "geocoded_boundingbox": result.get("boundingbox"),
        }

    def geocode_extracted_record(self, structured_params, cache=None):
        if cache is None:
            cache = self.load_nominatim_cache()
        geocode_output = {
            "nominatim_amenity": structured_params.get("amenity"),
            "nominatim_street": structured_params.get("street"),
            "nominatim_city": structured_params.get("city"),
            "nominatim_county": structured_params.get("county"),
            "nominatim_state": structured_params.get("state"),
            "nominatim_country": structured_params.get("country"),
            "nominatim_postalcode": structured_params.get("postalcode"),
            "nominatim_countrycodes": structured_params.get("countrycodes"),
            "geocode_status": "not_attempted",
        }
        if not structured_params:
            return None
    
        try:
            response_data = self.nominatim_search_structured(
                structured_params,
                limit=3,
                cache=cache,
            )
            results = response_data.get("results", [])
            geocode_output["geocode_status"] = "matched" if results else "no_match"
            geocode_output["nominatim_result_count"] = len(results)
            geocode_output.update(self.normalize_nominatim_result(results[0]))
    
            return geocode_output
    
        except Exception as e:
            import traceback
            traceback.print_exc()
            return None



In [10]:
PDF_processer = ProcessPDF()
prompt_builder = PromptBuilder()
llm_flow = LLMFlow(model, tokenizer)
response_parser = ResponseParser()
location_grounding = LocationGrounding()

In [11]:
class DataCenter:
    def __init__(self):
        self.data_center_specs = DataCenterSpecsExtraction().model_dump()
        self.data_center_location = DataCenterLocation().model_dump()
        self.location_cache = location_grounding.load_nominatim_cache()

    def get_specs(self, pdf_path: str):
        text = PDF_processer.extract_text(pdf_path)
        if not text: return self.data_center_specs
        chunks = PDF_processer.chunk_text(text)
        chunk_results = []
        for chunk in chunks:
            prompt = prompt_builder.extract_data_center_specs(chunk)
            extracted = llm_flow.extract_fields_from_text(prompt)
            parsed = response_parser.json_from_model_output(extracted, self.data_center_specs)
            chunk_results.append(parsed)
        for key in self.data_center_specs:
            self.data_center_specs[key] = response_parser.choose_best_value(
                [item.get(key) for item in chunk_results]
            )
        return self.data_center_specs
            
    def get_location(self, address):
        prompt = prompt_builder.nominatim_param(address)
        decoded = llm_flow.extract_params_from_address_line(prompt)
        parsed = response_parser.json_from_model_output(decoded)
        params = response_parser.create_nominatim_params(parsed, location_grounding.ALLOWED_NOMINATIM_QUERY_PARAMS)
        # TODO add checks for postal codes: for CA should be 6symbols (+space)
        params.pop('postalcode', None)
        # return self.data_center_location
        return location_grounding.geocode_extracted_record(params, self.location_cache)

In [12]:
class Material:
    MATERIAL_HEAT_CAPACITY_TABLE = [
        {"matched_material": "Air (sea level, dry, 0 C)", "phase": "gas", "specific_heat_kj_per_kg_k": 1.0035},
        {"matched_material": "Air (typical room conditions)", "phase": "gas", "specific_heat_kj_per_kg_k": 1.012},
        {"matched_material": "Aluminum", "phase": "solid", "specific_heat_kj_per_kg_k": 0.897},
        {"matched_material": "Ammonia", "phase": "liquid", "specific_heat_kj_per_kg_k": 4.700},
        {"matched_material": "Animal tissue", "phase": "mixed", "specific_heat_kj_per_kg_k": 3.500},
        {"matched_material": "Antimony", "phase": "solid", "specific_heat_kj_per_kg_k": 0.207},
        {"matched_material": "Argon", "phase": "gas", "specific_heat_kj_per_kg_k": 0.5203},
        {"matched_material": "Arsenic", "phase": "solid", "specific_heat_kj_per_kg_k": 0.328},
        {"matched_material": "Beryllium", "phase": "solid", "specific_heat_kj_per_kg_k": 1.820},
        {"matched_material": "Bismuth", "phase": "solid", "specific_heat_kj_per_kg_k": 0.123},
        {"matched_material": "Cadmium", "phase": "solid", "specific_heat_kj_per_kg_k": 0.231},
        {"matched_material": "Carbon dioxide", "phase": "gas", "specific_heat_kj_per_kg_k": 0.839},
        {"matched_material": "Chromium", "phase": "solid", "specific_heat_kj_per_kg_k": 0.449},
        {"matched_material": "Copper", "phase": "solid", "specific_heat_kj_per_kg_k": 0.385},
        {"matched_material": "Diamond", "phase": "solid", "specific_heat_kj_per_kg_k": 0.509},
        {"matched_material": "Ethanol", "phase": "liquid", "specific_heat_kj_per_kg_k": 2.440},
        {"matched_material": "Gasoline (octane)", "phase": "liquid", "specific_heat_kj_per_kg_k": 2.220},
        {"matched_material": "Glass", "phase": "solid", "specific_heat_kj_per_kg_k": 0.840},
        {"matched_material": "Gold", "phase": "solid", "specific_heat_kj_per_kg_k": 0.129},
        {"matched_material": "Granite", "phase": "solid", "specific_heat_kj_per_kg_k": 0.790},
        {"matched_material": "Graphite", "phase": "solid", "specific_heat_kj_per_kg_k": 0.710},
        {"matched_material": "Helium", "phase": "gas", "specific_heat_kj_per_kg_k": 5.193},
        {"matched_material": "Hydrogen", "phase": "gas", "specific_heat_kj_per_kg_k": 14.300},
        {"matched_material": "Hydrogen sulfide", "phase": "gas", "specific_heat_kj_per_kg_k": 1.015},
        {"matched_material": "Iron", "phase": "solid", "specific_heat_kj_per_kg_k": 0.449},
        {"matched_material": "Lead", "phase": "solid", "specific_heat_kj_per_kg_k": 0.129},
        {"matched_material": "Lithium", "phase": "solid", "specific_heat_kj_per_kg_k": 3.580},
        {"matched_material": "Lithium at 181 C", "phase": "solid(?)", "specific_heat_kj_per_kg_k": 4.233},
        {"matched_material": "Lithium at 181 C", "phase": "liquid", "specific_heat_kj_per_kg_k": 4.379},
        {"matched_material": "Magnesium", "phase": "solid", "specific_heat_kj_per_kg_k": 1.020},
        {"matched_material": "Mercury", "phase": "liquid", "specific_heat_kj_per_kg_k": 0.1395},
        {"matched_material": "Methane at 2 C", "phase": "gas", "specific_heat_kj_per_kg_k": 2.191},
        {"matched_material": "Methanol", "phase": "liquid", "specific_heat_kj_per_kg_k": 2.140},
        {"matched_material": "Molten salt", "phase": "liquid", "specific_heat_kj_per_kg_k": 1.560},
        {"matched_material": "Nitrogen", "phase": "gas", "specific_heat_kj_per_kg_k": 1.040},
        {"matched_material": "Neon", "phase": "gas", "specific_heat_kj_per_kg_k": 1.030},
        {"matched_material": "Oxygen", "phase": "gas", "specific_heat_kj_per_kg_k": 0.918},
        {"matched_material": "Paraffin wax", "phase": "solid", "specific_heat_kj_per_kg_k": 2.500},
        {"matched_material": "Polyethylene", "phase": "solid", "specific_heat_kj_per_kg_k": 2.302},
        {"matched_material": "Silica (fused)", "phase": "solid", "specific_heat_kj_per_kg_k": 0.703},
        {"matched_material": "Silver", "phase": "solid", "specific_heat_kj_per_kg_k": 0.233},
        {"matched_material": "Sodium", "phase": "solid", "specific_heat_kj_per_kg_k": 1.230},
        {"matched_material": "Steel", "phase": "solid", "specific_heat_kj_per_kg_k": 0.466},
        {"matched_material": "Tin", "phase": "solid", "specific_heat_kj_per_kg_k": 0.227},
        {"matched_material": "Titanium", "phase": "solid", "specific_heat_kj_per_kg_k": 0.523},
        {"matched_material": "Tungsten", "phase": "solid", "specific_heat_kj_per_kg_k": 0.134},
        {"matched_material": "Uranium", "phase": "solid", "specific_heat_kj_per_kg_k": 0.116},
        {"matched_material": "Water at 100 C (steam)", "phase": "gas", "specific_heat_kj_per_kg_k": 2.030},
        {"matched_material": "Water at 25 C", "phase": "liquid", "specific_heat_kj_per_kg_k": 4.181},
        {"matched_material": "Water at 100 C", "phase": "liquid", "specific_heat_kj_per_kg_k": 4.216},
        {"matched_material": "Water at -10 C (ice)", "phase": "solid", "specific_heat_kj_per_kg_k": 2.050},
        {"matched_material": "Zinc", "phase": "solid", "specific_heat_kj_per_kg_k": 0.387},
        {"matched_material": "Asphalt", "phase": "solid", "specific_heat_kj_per_kg_k": 0.920},
        {"matched_material": "Brick", "phase": "solid", "specific_heat_kj_per_kg_k": 0.840},
        {"matched_material": "Concrete", "phase": "solid", "specific_heat_kj_per_kg_k": 0.880},
        {"matched_material": "Glass, silica", "phase": "liquid", "specific_heat_kj_per_kg_k": 0.840},
        {"matched_material": "Glass, crown", "phase": "liquid", "specific_heat_kj_per_kg_k": 0.670},
        {"matched_material": "Glass, flint", "phase": "liquid", "specific_heat_kj_per_kg_k": 0.503},
        {"matched_material": "Glass, borosilicate", "phase": "liquid", "specific_heat_kj_per_kg_k": 0.753},
        {"matched_material": "Gypsum", "phase": "solid", "specific_heat_kj_per_kg_k": 1.090},
        {"matched_material": "Marble, mica", "phase": "solid", "specific_heat_kj_per_kg_k": 0.880},
        {"matched_material": "Sand", "phase": "solid", "specific_heat_kj_per_kg_k": 0.835},
        {"matched_material": "Soil", "phase": "solid", "specific_heat_kj_per_kg_k": 0.800},
        {"matched_material": "Water", "phase": "liquid", "specific_heat_kj_per_kg_k": 4.181},
        {"matched_material": "Wood", "phase": "solid", "specific_heat_kj_per_kg_k": 1.700},
    ]

    MATERIAL_ALIASES = {
        "reinforced concrete": "concrete",
        "structural concrete": "concrete",
        "cement": "concrete",
        "structural steel": "steel",
        "steel frame": "steel",
        "steel framing": "steel",
        "brickwork": "brick",
        "masonry brick": "brick",
        "silica glass": "glass silica",
        "crown glass": "glass crown",
        "flint glass": "glass flint",
        "borosilicate glass": "glass borosilicate",
        "ice": "water at 10 c ice",
        "steam": "water at 100 c steam",
    }

    def __init__(self, model=None, tokenizer=None, info_table_path="/kaggle/input/datasets/katerynazakharova/dc-specs/table_of_specific_heat_capacities.pdf"):
        self.model = model
        self.tokenizer = tokenizer

        self.material_thermal_capacity = MaterialThermalCapacityExtraction().model_dump()

        self.info_table_path = info_table_path
        self.pdf_processor = globals().get("PDF_processer", ProcessPDF())
        self.prompt_builder = globals().get("prompt_builder", PromptBuilder())
        self.response_parser = globals().get("response_parser", ResponseParser())
        self.llm_flow = LLMFlow(model, tokenizer) if model is not None and tokenizer is not None else globals().get("llm_flow")

        self.material_reference_table = pd.DataFrame(self.MATERIAL_HEAT_CAPACITY_TABLE)
        self.material_reference_lookup = {
            self.normalize_material_name(row["matched_material"]): row
            for row in self.MATERIAL_HEAT_CAPACITY_TABLE
        }

    def normalize_material_name(self, material):
        if material is None:
            return ""
        material = str(material).lower()
        material = material.replace("(", " ").replace(")", " ")
        material = re.sub(r"[^a-z0-9]+", " ", material)
        return re.sub(r"\s+", " ", material).strip()

    def default_thermal_capacity_results(self, materials):
        return [
            ThermalCapacity(material=material).model_dump()
            for material in materials
        ]

    def parse_specific_heat_value(self, value):
        if value is None:
            return None
        if isinstance(value, (int, float)):
            return float(value)
        match = re.search(r"[-+]?\d*\.?\d+", str(value).replace(",", ""))
        if not match:
            return None
        return float(match.group(0))

    def records_from_parsed_response(self, parsed):
        if isinstance(parsed, dict):
            if isinstance(parsed.get("thermal_capacities"), list):
                return parsed.get("thermal_capacities")
            if isinstance(parsed.get("materials"), list):
                return parsed.get("materials")
            if "material" in parsed:
                return [parsed]
        if isinstance(parsed, list):
            return parsed
        return []

    def merge_thermal_capacity_results(self, materials, chunk_results):
        results = self.default_thermal_capacity_results(materials)
        results_by_material = {
            self.normalize_material_name(item["material"]): item
            for item in results
        }
        for parsed in chunk_results:
            for record in self.records_from_parsed_response(parsed):
                if not isinstance(record, dict):
                    continue
                material_key = self.normalize_material_name(record.get("material"))
                if material_key not in results_by_material:
                    continue
                result = results_by_material[material_key]
                value = self.parse_specific_heat_value(record.get("specific_heat_kj_per_kg_k"))
                if value is not None:
                    result["specific_heat_kj_per_kg_k"] = value
                if record.get("matched_material"):
                    result["matched_material"] = record.get("matched_material")
                if record.get("phase"):
                    result["phase"] = record.get("phase")
        return results

    def lookup_material_reference(self, material):
        material_key = self.normalize_material_name(material)
        alias_key = self.MATERIAL_ALIASES.get(material_key, material_key)

        if alias_key in self.material_reference_lookup:
            return self.material_reference_lookup[alias_key]

        for reference_key, reference_record in self.material_reference_lookup.items():
            if alias_key and (alias_key == reference_key or alias_key in reference_key or reference_key in alias_key):
                return reference_record
        return None

    def add_reference_values(self, material_results):
        for result in material_results:
            if result.get("specific_heat_kj_per_kg_k") is not None:
                continue
            reference = self.lookup_material_reference(result.get("material"))
            if reference is None:
                continue
            result["matched_material"] = reference["matched_material"]
            result["phase"] = reference["phase"]
            result["specific_heat_kj_per_kg_k"] = reference["specific_heat_kj_per_kg_k"]
        return material_results

    def get_thermal_capacity(self, materials):
        if not materials:
            return []

        text = ""
        if Path(self.info_table_path).exists():
            text = self.pdf_processor.extract_text(self.info_table_path)

        chunk_results = []
        if text and self.llm_flow is not None:
            chunks = self.pdf_processor.chunk_text(text)
            for chunk in chunks:
                prompt = self.prompt_builder.extract_material_thermal_capacity(chunk, materials)
                extracted = self.llm_flow.extract_fields_from_text(prompt, max_new_tokens=512)
                parsed = self.response_parser.json_from_model_output(extracted, self.material_thermal_capacity)
                chunk_results.append(parsed)

        material_results = self.merge_thermal_capacity_results(materials, chunk_results)
        return self.add_reference_values(material_results)

In [14]:
single_pdf = "/kaggle/input/datasets/katerynazakharova/dc-specs/ibx_tr1_en.pdf"
DC = DataCenter()
dc_specs = DC.get_specs(single_pdf)
print(dc_specs)
dc_location = DC.get_location({"address": dc_specs["address"]})

['{\n "data_center_name": "TR1 Toronto IBX® Data Center",\n "address": "151 Front Street West 3rd, 5th and 7th floors Toronto, Ontario M5 2N1 Canada",\n "colocation_space_sqm": 3352,\n "building_materials": [\n  "concrete",\n  "steel"\n ],\n "maximum_power_load_kw": null,\n "standby_power_total_kw": 6500,\n "cooling_configuration": "Chilled water, DX systems and glycol systems",\n "notes": [\n  "8-story building",\n  "Raised floor",\n  "Flood Zone D",\n  "Seismic Design Category X"\n ]\n}']
{'data_center_name': 'TR1 Toronto IBX® Data Center', 'address': '151 Front Street West 3rd, 5th and 7th floors Toronto, Ontario M5 2N1 Canada', 'colocation_space_sqm': '3352', 'building_materials': "['concrete', 'steel']", 'maximum_power_load_kw': None, 'standby_power_total_kw': '6500', 'cooling_configuration': 'Chilled water, DX systems and glycol systems', 'notes': "['8-story building', 'Raised floor', 'Flood Zone D', 'Seismic Design Category X']"}
['```json\n{\n  "amenity": null,\n  "street": "15

In [15]:
M = Material()
thermal_capacity_list = M.get_thermal_capacity(eval(dc_specs["building_materials"]))

In [16]:
thermal_capacity_list

[{'material': 'concrete',
  'matched_material': 'Concrete',
  'phase': 'solid',
  'specific_heat_kj_per_kg_k': 0.88},
 {'material': 'steel',
  'matched_material': 'Steel',
  'phase': 'solid',
  'specific_heat_kj_per_kg_k': 0.466}]

In [192]:
dc_location

{'nominatim_amenity': None,
 'nominatim_street': '151 Front Street West',
 'nominatim_city': 'Toronto',
 'nominatim_county': None,
 'nominatim_state': 'Ontario',
 'nominatim_country': 'Canada',
 'nominatim_postalcode': None,
 'nominatim_countrycodes': 'ca',
 'geocode_status': 'matched',
 'nominatim_result_count': 4,
 'latitude': '43.6446634',
 'longitude': '-79.3841764',
 'geocoded_display_name': '151, Front Street West, Wellington Place, Spadina—Fort York, Toronto, Golden Horseshoe, Ontario, M5V 2X3, Canada',
 'geocoded_class': None,
 'geocoded_type': 'yes',
 'geocoded_importance': 8.130235751883753e-05,
 'geocoded_osm_type': 'way',
 'geocoded_osm_id': 43804185,
 'geocoded_place_id': 325423829,
 'geocoded_boundingbox': ['43.6443845',
  '43.6449422',
  '-79.3847981',
  '-79.3835547']}

## Run Load estimator
Works with an example code. Uses H100 GPU specs for the math.
Upon facing any uncertainties, asks the user for clarifications. There will be input for the user to insert the number of GPUs required. 

In [18]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer, 
    chat_template="gemma-4-thinking"
)

In [19]:
def read_user_code(directory_path):
    code_contents = []
    for root, _, files in os.walk(directory_path):
        for file in files:
            if file.endswith((".py", ".sh", ".yaml")):
                file_path = os.path.join(root, file)
                try:
                    with open(file_path, "r", encoding="utf-8") as f:
                        content = f.read()
                        code_contents.append(f"--- File: {file_path} ---\n{content}\n")
                except Exception as e:
                    print(f"Could not read {file_path}: {e}")
    return "\n".join(code_contents)

SYSTEM_PROMPT = """You are an expert AI data center architect and power estimator.
Your goal is to estimate the instantaneous power load (in Megawatts - MW) required to run the provided machine learning code.

Hardware assumptions:
- The code will be run on NVIDIA H100 SXM GPUs.
- The quantity of GPUs will be between 1 and 10.
- 1x H100 SXM GPU draws approximately 700W (0.0007 MW) under peak training load.
- Assume standard host system overhead (CPU, RAM, networking) adds roughly 1000W (0.001 MW) per 4-GPU block.
- Assume a data center Power Usage Effectiveness (PUE) of 1.2.

Step 1: Analyze the provided code.
Step 2: Ask the user for VERIFICATION (exact number of GPUs, dataset size, epochs).
Step 3: Once the user provides the missing details, calculate the total peak power load in Megawatts (MW). 
Step 4: Once you have the final number, you MUST output the final answer STRICTLY as a JSON object exactly like this:
{"load_estimate": <float>, "unit": "MW"}
Do not include any other text in your final response after outputting the JSON.
"""

def chat_with_estimator(code_dir=None):
    if not code_dir:
        code_dir = input("Enter the path to your working directory: ")
    user_code = read_user_code(code_dir)
    
    if not user_code.strip():
        print("No valid code files found in that directory. Exiting.")
        return None

    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [{"type": "text", "text": f"Here is my training code. Please analyze it and let me know what you need to calculate the MW load:\n\n{user_code}"}]}
    ]

    print("\n--- Starting Estimation Session (Type 'exit' to quit) ---\n")
    
    while True:
        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt"
        ).to("cuda")
        
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=1024,
            use_cache=True,
            temperature=0.3, 
        )
        
        prompt_length = inputs.shape[1]
        response_text = tokenizer.decode(outputs[0][prompt_length:], skip_special_tokens=True)
        
        print(f"\nGemma Estimator:\n{response_text}\n")
        json_match = re.search(r'\{.*"load_estimate".*\}', response_text, re.DOTALL | re.IGNORECASE)
        
        if json_match:
            try:
                final_estimate_dict = json.loads(json_match.group(0))
                return final_estimate_dict
            except json.JSONDecodeError:
                print("JSON parsing error")

        messages.append({"role": "model", "content": [{"type": "text", "text": response_text}]})
        
        user_input = input("You (Verification/Details): ")
        if user_input.lower() in ['exit', 'quit']:
            break
            
        messages.append({"role": "user", "content": [{"type": "text", "text": user_input}]})

In [20]:
result = chat_with_estimator(code_dir='/kaggle/input/datasets/katerynazakharova/ml-code/code/sample1')

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



--- Starting Estimation Session (Type 'exit' to quit) ---


Gemma Estimator:
I have analyzed your code. You are performing a PEFT (LoRA) fine-tuning of the **Gemma-2-27b-it** model using 4-bit quantization (`BitsAndBytesConfig`) and the `SFTTrainer` from the TRL library.

To provide an accurate instantaneous power load estimate in Megawatts (MW), I need you to verify the following details:

1.  **Number of GPUs:** How many NVIDIA H100 SXM GPUs will be used for this training run? (Between 1 and 10).
2.  **Dataset Size:** I see `num_dataset_samples: 1000` in your `config.py`. Is this the final size you intend to use?
3.  **Epochs:** I see `num_epochs: 3` in your `config.py`. Please confirm if this is correct.

Once you provide these details, I will calculate the peak power load including host overhead and PUE.



You (Verification/Details):  1 GPU



Gemma Estimator:
```json
{"load_estimate": 0.00204, "unit": "MW"}
```



In [22]:
result

{'load_estimate': 0.00204, 'unit': 'MW'}

## Run Orchestrator Model
Each local agent will either accept or reject the suggested workload. 

In [24]:
from dataclasses import asdict, dataclass, field
from pathlib import Path
from PIL import Image
from typing import Any, Dict, List, Literal, Optional, Tuple

PER_SITE_FLEET_CAP_MW = 150.0
Verdict = Literal["accept", "reject"]
PLUME_CONCERN_TAGS = frozenset(
    {
        "plume_too_local",
        "plume_too_local_or_intense",
        "localized_hotspot",
        "vegetation_exposure",
        "downwind_spread_risk",
        "large_hot_footprint",
        "residential_exposure",
    }
)

@dataclass
class DatacenterThresholds:
    max_delta_t_with_concerns_c: float = 9.5
    plume_too_local_hot2_cells: float = 400.0
    plume_too_local_central_c: float = 3.0
    large_hot_footprint_hot5_cells: float = 100.0
    large_hot_footprint_central_c: float = 10.0
    downwind_max_anomaly_c: float = 8.0
    downwind_min_extra_mw: float = 0.5

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


@dataclass
class SiteReview:
    site_index: int
    site_name: str
    load_mw: float
    verdict: Verdict
    proposed_max_load_mw: Optional[float]
    reasons: List[str] = field(default_factory=list)
    concerns: List[str] = field(default_factory=list)
    source: str = "heuristic"
    central_delta_t_c: float = 0.0
    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


@dataclass
class ObjectiveContext:
    site_weights: List[float]
    max_loads_mw: List[float]
    round_index: int = 0
    orchestrator_notes: str = ""
    rejected_site_indices: List[int] = field(default_factory=list)
    per_site_fleet_cap_mw: float = PER_SITE_FLEET_CAP_MW
    target_total_mw: Optional[float] = None

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

def _central_delta_t(metrics: Dict[str, Any]) -> float:
    return 0.0


def _max_delta_t(metrics: Dict[str, Any]) -> float:
    c = _central_delta_t(metrics)
    return float(c)


def _detect_plume_concerns(metrics: Dict[str, Any], thresholds: DatacenterThresholds) -> List[str]:
    central = _central_delta_t(metrics)
    max_d = _max_delta_t(metrics)
    hot2 = 0.0
    hot5 = 0.0

    concerns: List[str] = []
    if hot5 > thresholds.large_hot_footprint_hot5_cells or central > thresholds.large_hot_footprint_central_c:
        concerns.append("large_hot_footprint")
    if hot2 > thresholds.plume_too_local_hot2_cells and central > thresholds.plume_too_local_central_c:
        concerns.append("plume_too_local")
    if max_d > thresholds.downwind_max_anomaly_c:
        concerns.append("downwind_spread_risk")
    return concerns


def _normalize_concerns(raw: List[str]) -> List[str]:
    out: List[str] = []
    for tag in raw:
        t = str(tag).strip().lower().replace(" ", "_")
        if not t:
            continue
        if t in PLUME_CONCERN_TAGS or t.startswith("plume") or "vegetation" in t or "local" in t:
            out.append(t)
    return list(dict.fromkeys(out))


def _apply_threshold_policy(
    *,
    site_index: int,
    site_name: str,
    load_mw: float,
    base_load_mw: float,
    metrics: Dict[str, Any],
    thresholds: DatacenterThresholds,
    concern_tags: List[str],
    source: str,
    llm_proposed_cap: Optional[float] = None,
) -> SiteReview:
    """
    Final verdict:
    - No concerns → accept, keep loads (no cap).
    - Concerns + central ΔT > limit → reject, lower cap.
    - Concerns + central ΔT ≤ limit → accept (plume flagged but within policy).
    """
    central = _central_delta_t(metrics)
    extra = load_mw - base_load_mw
    concerns = _normalize_concerns(concern_tags)
    if not concerns:
        concerns = _detect_plume_concerns(metrics, thresholds)

    limit = float(thresholds.max_delta_t_with_concerns_c)
    reasons: List[str] = []
    cap: Optional[float] = None
    verdict: Verdict

    if not concerns:
        verdict = "accept"
        reasons.append(
            f"Plume OK for {site_name}: no concerns flagged; keeping {load_mw:.2f} MW "
            f"(central ΔT {central:.2f} °C)."
        )
    elif central > limit:
        verdict = "reject"
        reasons.append(
            f"Concerns {concerns}: central ΔT {central:.2f} °C exceeds limit "
            f"{limit:.1f} °C — reduce load."
        )
        cap = llm_proposed_cap
        if cap is None:
            cap = max(base_load_mw, load_mw - max(0.5, 0.25 * max(extra, 0.5)))
    else:
        verdict = "accept"
        reasons.append(
            f"Concerns {concerns} noted, but central ΔT {central:.2f} °C ≤ {limit:.1f} °C "
            f"— load {load_mw:.2f} MW acceptable; no change."
        )

    return SiteReview(
        site_index=site_index,
        site_name=site_name,
        load_mw=load_mw,
        verdict=verdict,
        proposed_max_load_mw=cap,
        reasons=reasons,
        concerns=concerns,
        source=source,
        central_delta_t_c=central,
    )


def _text_from_response(response: Any) -> str:
    t = getattr(response, "text", None)
    if isinstance(t, str) and t.strip():
        return t
    cands = getattr(response, "candidates", None) or []
    if cands:
        content = getattr(cands[0], "content", None)
        parts = getattr(content, "parts", None) if content else None
        if parts:
            return "".join(str(getattr(p, "text", "") or "") for p in parts)
    return ""


def _parse_json_blob(text: str) -> Dict[str, Any]:
    text = (text or "").strip()
    m = re.search(r"\{[\s\S]*\}", text)
    if not m:
        return {}
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return {}



def _load_image_part(path: Path, as_tensor: bool = False):
    image = Image.open(path).convert("RGB")
    if as_tensor:
        transform = transforms.Compose([transforms.ToTensor()])
        return transform(image)
    return image


def _is_reykjavik_site(site_index: int, site_name: str) -> bool:
    return site_index == 1 or "reykjavik" in site_name.lower()


def _rule_based_site_review(
    site_index: int,
    site_name: str,
    load_mw: float,
    base_load_mw: float,
    metrics: Dict[str, Any],
    thresholds: DatacenterThresholds,
    agent_round: int = 0,
) -> SiteReview:
    """
    Rule-based local agent.

    Reykjavik rejects once on the first review (downwind spread risk); after
    redistribution it accepts. Other sites always accept in rule-based mode.
    """
    central = _central_delta_t(metrics)

    if _is_reykjavik_site(site_index, site_name) and agent_round == 0:
        concerns = ["downwind_spread_risk"]
        extra = load_mw - base_load_mw
        cap = max(base_load_mw, load_mw - max(0.5, 0.25 * max(extra, 0.5)))
        return SiteReview(
            site_index=site_index,
            site_name=site_name,
            load_mw=load_mw,
            verdict="reject",
            proposed_max_load_mw=cap,
            reasons=[
                f"{site_name}: downwind spread risk — reduce load "
                f"(cap → {cap:.2f} MW, central ΔT {central:.2f} °C)."
            ],
            concerns=concerns,
            source="rules",
            central_delta_t_c=central,
        )

    return SiteReview(
        site_index=site_index,
        site_name=site_name,
        load_mw=load_mw,
        verdict="accept",
        proposed_max_load_mw=None,
        reasons=[
            f"{site_name} accepted — plume OK (central ΔT {central:.2f} °C)."
        ],
        concerns=[],
        source="rules",
        central_delta_t_c=central,
    )


def _llm_site_review(
    *,
    datacenter: Dict[str, Any],
    site_index: int,
    load_mw: float,
    base_load_mw: float,
    metrics: Dict[str, Any],
    thresholds: DatacenterThresholds,
    anomaly_png: Optional[Path],
    plume_gif: Optional[Path],
    final_temp_png: Optional[Path],
    agent_round: int = 0,
) -> SiteReview:
    name = str(datacenter.get("name", f"site_{site_index}"))
    #if not key:
    #   return _rule_based_site_review(
     #       site_index, name, load_mw, base_load_mw, metrics, thresholds, agent_round=agent_round
     #   )

    parts: List[Any] = []
    if anomaly_png and anomaly_png.is_file():
        parts.append(_load_image_part(anomaly_png))
    if final_temp_png and final_temp_png.is_file():
        parts.append(_load_image_part(final_temp_png))
    if plume_gif and plume_gif.is_file():
        parts.append(_load_image_part(plume_gif))

    limit = thresholds.max_delta_t_with_concerns_c
    metrics_blob = json.dumps(metrics, indent=2, default=str)[:6000]
    prompt = f"""You are the on-site operations agent for datacenter "{name}".

Assigned load: **{load_mw:.3f} MW** (baseline {base_load_mw:.3f} MW).
Central building ΔT above ambient is in metrics as ``central_building_anomaly_C``.

Tag plume **concerns** only when warranted, e.g.:
- ``plume_too_local`` — heat too concentrated on the central building
- ``vegetation_exposure`` — plume toward parks / vegetation
- ``downwind_spread_risk`` — anomaly extending far downwind
- ``large_hot_footprint`` — very large hot area

Do **not** invent concerns if the plume looks acceptable.

Policy (enforced server-side after your reply):
- **No concerns** → loads stay unchanged (accept).
- **Any concerns** → central ΔT must be ≤ {limit:.1f} °C or the site must reject and lower MW.

Metrics JSON:
{metrics_blob}

Respond with **JSON only**:
{{
  "concerns": ["plume_too_local", ...] or [],
  "proposed_max_load_mw": number or null,
  "notes": "short rationale"
}}
"""
    # parts.append(prompt)
    resp = llm_flow.model_inference_image(prompt, parts, 1024)
    data = _parse_json_blob(_text_from_response(resp))
    concerns = [str(x) for x in (data.get("concerns") or [])]
    cap = data.get("proposed_max_load_mw")
    cap_f = float(cap) if cap is not None else None
    review = _apply_threshold_policy(
        site_index=site_index,
        site_name=name,
        load_mw=load_mw,
        base_load_mw=base_load_mw,
        metrics=metrics,
        thresholds=thresholds,
        concern_tags=concerns,
        source="llm",
        llm_proposed_cap=cap_f,
    )
    if data.get("notes"):
        review.reasons.insert(0, str(data["notes"])[:500])
    return review


def review_site_local(
    *,
    datacenter: Dict[str, Any],
    site_index: int,
    load_mw: float,
    base_load_mw: float,
    metrics: Dict[str, Any],
    anomaly_png: Optional[Path],
    plume_gif: Optional[Path],
    final_temp_png: Optional[Path] = None,
    use_llm: bool = True,
    thresholds: Optional[DatacenterThresholds] = None,
    agent_round: int = 0,
) -> SiteReview:
    """Local datacenter agent: rule-based or vision LLM, then threshold policy."""
    name = str(datacenter.get("name", f"site_{site_index}"))
    th = thresholds or DatacenterThresholds()

    if not use_llm:
        return _rule_based_site_review(
            site_index, name, load_mw, base_load_mw, metrics, th, agent_round=agent_round
        )

    try:
        return _llm_site_review(
            datacenter=datacenter,
            site_index=site_index,
            load_mw=load_mw,
            base_load_mw=base_load_mw,
            metrics=metrics,
            thresholds=th,
            anomaly_png=anomaly_png,
            plume_gif=plume_gif,
            final_temp_png=final_temp_png,
            agent_round=agent_round,
        )
    except Exception as exc:
        print("Local agent %s LLM failed (%s); rule-based fallback", name, exc)
        return _rule_based_site_review(
            site_index, name, load_mw, base_load_mw, metrics, th, agent_round=agent_round
        )


def _rule_based_orchestrator_adjust(
    reviews: List[SiteReview],
    base_loads_mw: List[float],
    current_loads_mw: List[float],
    min_loads_mw: List[float],
    max_loads_mw: List[float],
) -> Tuple[List[float], List[float], str, List[int]]:
    n = len(reviews)
    weights = [1.0] * n
    new_max = [
        max(float(min_loads_mw[i]), PER_SITE_FLEET_CAP_MW) for i in range(n)
    ]
    rejected_idx: List[int] = []
    notes: List[str] = []

    for r in reviews:
        if r.verdict != "reject":
            continue
        i = r.site_index
        rejected_idx.append(i)
        weights[i] = min(6.0, weights[i] * 2.8)
        cap = r.proposed_max_load_mw
        if cap is None:
            cap = max(base_loads_mw[i], current_loads_mw[i] - 1.0)
        new_max[i] = min(new_max[i], max(min_loads_mw[i], float(cap)))
        notes.append(f"{r.site_name}: cap → {new_max[i]:.2f} MW")

    accepted = sum(1 for r in reviews if r.verdict == "accept")
    if accepted:
        notes.append(f"{accepted} site(s) accepted — no load change on those sites.")
    return (
        weights,
        new_max,
        " | ".join(notes) if notes else "No rejected sites.",
        rejected_idx,
    )


def orchestrate_objective(
    *,
    datacenters: List[Dict[str, Any]],
    reviews: List[SiteReview],
    base_loads_mw: List[float],
    current_loads_mw: List[float],
    min_loads_mw: List[float],
    max_loads_mw: List[float],
    round_index: int,
    use_llm: bool = True,
) -> ObjectiveContext:
    """Translate rejections into weights/MW caps for the next BO pass."""
    n = len(datacenters)
    rejected = [r for r in reviews if r.verdict == "reject"]

    fleet_cap = PER_SITE_FLEET_CAP_MW

    if not rejected:
        return ObjectiveContext(
            site_weights=[1.0] * n,
            max_loads_mw=[
                max(float(min_loads_mw[i]), fleet_cap) for i in range(n)
            ],
            round_index=round_index,
            orchestrator_notes="All sites accepted — loads unchanged.",
            rejected_site_indices=[],
            per_site_fleet_cap_mw=fleet_cap,
        )

    weights, new_max, notes, rejected_idx = _rule_based_orchestrator_adjust(
        reviews, base_loads_mw, current_loads_mw, min_loads_mw, max_loads_mw
    )

    if use_llm:
        try:
            payload = {
                "round": round_index,
                "reviews": [r.to_dict() for r in reviews],
                "base_loads_mw": base_loads_mw,
                "current_loads_mw": current_loads_mw,
                "proposed_weights": weights,
                "proposed_max_loads_mw": new_max,
            }
            prompt = f"""You are the global load orchestrator.

Only adjust sites that **rejected** load. Accepted sites must keep their MW.
Redistribute excess MW to accepting sites via weights and caps.

{json.dumps(payload, indent=2, default=str)[:12000]}

JSON only:
{{"site_weights": [...], "max_loads_mw": [...], "notes": "..."}}
"""
            resp = llm_flow.model_inference(prompt, 2048)
            data = _parse_json_blob(_text_from_response(resp))
            if data.get("site_weights") and len(data["site_weights"]) == n:
                weights = [float(x) for x in data["site_weights"]]
            if data.get("max_loads_mw") and len(data["max_loads_mw"]) == n:
                for i in rejected_idx:
                    llm_cap = float(data["max_loads_mw"][i])
                    new_max[i] = max(
                        min_loads_mw[i],
                        min(new_max[i], llm_cap),
                    )
            if data.get("notes"):
                notes = str(data["notes"])
        except Exception as exc:
            print("Orchestrator LLM failed (%s); rule-based caps", exc)

    return ObjectiveContext(
        site_weights=weights,
        max_loads_mw=new_max,
        round_index=round_index,
        orchestrator_notes=notes,
        rejected_site_indices=rejected_idx,
        per_site_fleet_cap_mw=fleet_cap,
    )


def review_all_sites(
    datacenters: List[Dict[str, Any]],
    final_results: List[Dict[str, Any]],
    base_loads_mw: List[float],
    current_loads_mw: List[float],
    *,
    use_llm: bool = True,
    thresholds: Optional[DatacenterThresholds] = None,
    agent_round: int = 0,
) -> Tuple[List[SiteReview], bool]:
    reviews: List[SiteReview] = []
    th = thresholds or DatacenterThresholds()
    for i, (dc, res) in enumerate(zip(datacenters, final_results)):
        od = Path(res["output_dir"])
        metrics = res.get("metrics") or res
        review = review_site_local(
            datacenter=dc,
            site_index=i,
            load_mw=float(current_loads_mw[i]),
            base_load_mw=float(base_loads_mw[i]),
            metrics=metrics,
            anomaly_png=od / "03_final_anomaly.png",
            plume_gif=od / "04_heat_plume_animation.gif",
            final_temp_png=od / "02_final_temperature.png",
            use_llm=use_llm,
            thresholds=th,
            agent_round=agent_round,
        )
        reviews.append(review)
    all_ok = all(r.verdict == "accept" for r in reviews)
    return reviews, all_ok


def objective_context_to_dict(ctx: Optional[ObjectiveContext]) -> Optional[Dict[str, Any]]:
    if ctx is None:
        return None
    return ctx.to_dict()

In [25]:
reviews, all_ok = review_all_sites(
                [{'name': 'Toronto Edge DC', 'lat': 43.6532, 'lon': -79.3832, 'temp_c': 21.0, 'humidity': 58.0, 'solar_wm2': 620.0, 'wind_speed_m_s': 4.2, 'wind_direction': 'NE', 'wall_material': 'Concrete (built-in)', 'wall_specific_heat_kj_per_kg_k': 0.88, 'wall_density_kg_m3': 2400.0}, {'name': 'Reykjavik Cold DC', 'lat': 64.1466, 'lon': -21.9426, 'temp_c': 8.0, 'humidity': 71.0, 'solar_wm2': 410.0, 'wind_speed_m_s': 0.2, 'wind_direction': 'W', 'wall_material': 'Concrete (built-in)', 'wall_specific_heat_kj_per_kg_k': 0.88, 'wall_density_kg_m3': 2400.0}, {'name': 'TR1 Toronto IBX Data Center', 'lat': 43.6446634, 'lon': -79.3841764, 'temp_c': 22.0, 'humidity': 55.0, 'solar_wm2': 600.0, 'wind_speed_m_s': 4.0, 'wind_direction': 'NE', 'wall_material': 'Steel', 'wall_specific_heat_kj_per_kg_k': 0.466}],
                [{'metrics': {'load': 1.36, 'load_mw': 68.0, 'Q_source_W_m2': 25962.416622277415, 'Q_central_solar_W_m2': 4.0920000000000005, 'source_heat_multiplier': 1.0, 'wind_x_m_s': -0.29698484809834996, 'wind_y_m_s': -0.2969848480983501, 'ambient_wind_speed_m_s': 4.2, 'mean_transport_wind_speed_m_s': 0.2600054296875, 'mean_convective_wind_speed_m_s': 3.3714105468750013, 'T_air_C': 21.0, 'humidity': 0.58, 'solar_radiation_W_m2': 620.0, 'pue_nominal': 1.35, 'air_density_kg_m3': 1.193533812598869, 'air_cp_J_kgK': 1012.6120461205417, 'air_areal_heat_capacity_J_m2K': 21754.560889616256, 'effective_sky_temperature_C': 7.670649595525958, 'mean_convective_h_W_m2K': 18.511360078125005, 'mean_exposure_fraction': 0.6331523437499998, 'dt_s': 0.2, 'steps': 3000, 'simulation_time_s': 600.0, 'simulation_time_min': 10.0, 'dx_m': 2.5, 'dy_m': 2.5, 'alpha_max_m2_s': 0.150022, 'alpha_mean_m2_s': 0.08046248302734373, 'stable_dt_limit_s': 9.998533548412901, 'central_building_temperature_C': 28.89829043914173, 'central_building_anomaly_C': 7.898290439141729, 'max_temp_C': 28.89829043914173, 'mean_temp_C': 21.47803278319837, 'max_anomaly_C': 7.898290439141729, 'mean_anomaly_C': 0.47803278319836756, 'total_anomaly_C_cells': 12237.63924987821, 'hot_area_gt_0p2C_cells': 3582, 'hot_area_gt_0p5C_cells': 3003, 'hot_area_gt_1C_cells': 2485, 'hot_area_gt_2C_cells': 1953, 'hot_area_gt_5C_cells': 1115, 'hot_area_gt_30C_cells': 0, 'hot_area_gt_35C_cells': 0, 'hot_area_gt_40C_cells': 0, 'hot_area_gt_45C_cells': 0, 'thermal_risk_objective': 28.89829043914173, 'building_cells': 14747, 'central_building_cells': 555, 'mask_source_area_m2': 3468.75, 'physical_roof_area_m2': 2531.0, 'source_area_m2': 3468.75, 'usable_power_density_W_m2': 987.7518767285658, 'source_heat_density_W_m2': 25962.416622277415, 'effective_source_heat_density_W_m2': 25962.416622277415, 'central_solar_heat_density_W_m2': 4.0920000000000005, 'window_to_wall_ratio': 0.08, 'glass_area_m2': 266.24, 'opaque_wall_area_m2': 3061.76, 'total_wall_area_m2': 3328.0, 'UA_total_W_K': 2376.1920000000005, 'H_envelope_areal_W_m2K': 0.9388352429869619, 'heat_rejection_fraction': 1.0, 'ventilation_Vdot_m3_s': 5.624444444444444, 'dc_indoor_setpoint_C': 24.0, 'Q_it_MW': 68.0, 'COP': 4.456931818181817, 'PUE': 1.324369597919482, 'Q_cooling_electric_MW': 15.257132658524773, 'Q_facility_MW': 90.05713265852478, 'Q_rejected_to_outdoor_MW': 90.05713265852478, 'Q_solar_opaque_MW': 0.941532, 'Q_solar_window_MW': 0.05777408, 'Q_solar_gain_MW': 0.99930608, 'Q_conduction_loss_MW': 0.009786126000000001, 'Q_ventilation_loss_MW': 0.020392886522821754, 'Q_convection_loss_MW': 1.002339972572294, 'Q_radiation_loss_MW': 0.6292712185893065, 'Q_passive_net_MW': -0.6624841236844221, 'T_sky_C': 7.670649595525958, 'h_external_W_m2K': 21.66}, 'output_dir': 'load_optimization_outputs/opt_37fe16bf751a47/agent_round_0/00_Toronto_Edge_DC_load_68.000MW'}, {'metrics': {'load': 1.272, 'load_mw': 63.6, 'Q_source_W_m2': 30433.293823258267, 'Q_central_solar_W_m2': 2.7060000000000004, 'source_heat_multiplier': 1.0, 'wind_x_m_s': 0.020000000000000004, 'wind_y_m_s': 6.123233995736767e-18, 'ambient_wind_speed_m_s': 0.2, 'mean_transport_wind_speed_m_s': 0.014660312500000008, 'mean_convective_wind_speed_m_s': 0.17146250000000002, 'T_air_C': 8.0, 'humidity': 0.71, 'solar_radiation_W_m2': 410.0, 'pue_nominal': 1.35, 'air_density_kg_m3': 1.2519101809371664, 'air_cp_J_kgK': 1009.0088267656457, 'air_areal_heat_capacity_J_m2K': 22737.391611900795, 'effective_sky_temperature_C': -9.565676974576434, 'mean_convective_h_W_m2K': 6.351557499999999, 'mean_exposure_fraction': 0.7349453125, 'dt_s': 0.2, 'steps': 3000, 'simulation_time_s': 600.0, 'simulation_time_min': 10.0, 'dx_m': 2.5, 'dy_m': 2.5, 'alpha_max_m2_s': 0.150022, 'alpha_mean_m2_s': 0.10083189588437498, 'stable_dt_limit_s': 9.998533548412901, 'central_building_temperature_C': 14.869033386681352, 'central_building_anomaly_C': 6.869033386681352, 'max_temp_C': 14.869033386681352, 'mean_temp_C': 8.04821735698146, 'max_anomaly_C': 6.869033386681352, 'mean_anomaly_C': 0.048217356981459555, 'total_anomaly_C_cells': 1234.3643387253646, 'hot_area_gt_0p2C_cells': 508, 'hot_area_gt_0p5C_cells': 392, 'hot_area_gt_1C_cells': 290, 'hot_area_gt_2C_cells': 205, 'hot_area_gt_5C_cells': 89, 'hot_area_gt_30C_cells': 0, 'hot_area_gt_35C_cells': 0, 'hot_area_gt_40C_cells': 0, 'hot_area_gt_45C_cells': 0, 'thermal_risk_objective': 14.869033386681352, 'building_cells': 10480, 'central_building_cells': 76, 'mask_source_area_m2': 475.0, 'physical_roof_area_m2': 2531.0, 'source_area_m2': 2531.0, 'usable_power_density_W_m2': 987.7518767285658, 'source_heat_density_W_m2': 30433.293823258267, 'effective_source_heat_density_W_m2': 30433.293823258267, 'central_solar_heat_density_W_m2': 2.7060000000000004, 'window_to_wall_ratio': 0.08, 'glass_area_m2': 266.24, 'opaque_wall_area_m2': 3061.76, 'total_wall_area_m2': 3328.0, 'UA_total_W_K': 2376.1920000000005, 'H_envelope_areal_W_m2K': 0.9388352429869619, 'heat_rejection_fraction': 1.0, 'ventilation_Vdot_m3_s': 5.624444444444444, 'dc_indoor_setpoint_C': 24.0, 'Q_it_MW': 63.6, 'COP': 9.0, 'PUE': 1.2111111111111112, 'Q_cooling_electric_MW': 7.066666666666666, 'Q_facility_MW': 77.02666666666667, 'Q_rejected_to_outdoor_MW': 77.02666666666667, 'Q_solar_opaque_MW': 0.622626, 'Q_solar_window_MW': 0.03820544, 'Q_solar_gain_MW': 0.66083144, 'Q_conduction_loss_MW': 0.05219267200000001, 'Q_ventilation_loss_MW': 0.11367572971796724, 'Q_convection_loss_MW': 0.25998700631717664, 'Q_radiation_loss_MW': 0.6143091948454222, 'Q_passive_net_MW': -0.37933316288056607, 'T_sky_C': -9.565676974576434, 'h_external_W_m2K': 6.46}, 'output_dir': 'load_optimization_outputs/opt_37fe16bf751a47/agent_round_0/01_Reykjavik_Cold_DC_load_63.600MW'}, {'metrics': {'load': 1.7479999999999998, 'load_mw': 87.39999999999999, 'Q_source_W_m2': 46085.05020900367, 'Q_central_solar_W_m2': 3.96, 'source_heat_multiplier': 1.0, 'wind_x_m_s': -0.282842712474619, 'wind_y_m_s': -0.28284271247461906, 'ambient_wind_speed_m_s': 4.0, 'mean_transport_wind_speed_m_s': 0.25582734375, 'mean_convective_wind_speed_m_s': 3.2332890625000004, 'T_air_C': 22.0, 'humidity': 0.55, 'solar_radiation_W_m2': 600.0, 'pue_nominal': 1.35, 'air_density_kg_m3': 1.1894375334551606, 'air_cp_J_kgK': 1012.6744317851116, 'air_areal_heat_capacity_J_m2K': 21681.23361004061, 'effective_sky_temperature_C': 8.698667035086828, 'mean_convective_h_W_m2K': 17.986498437499996, 'mean_exposure_fraction': 0.6438847656249999, 'dt_s': 0.2, 'steps': 3000, 'simulation_time_s': 600.0, 'simulation_time_min': 10.0, 'dx_m': 2.5, 'dy_m': 2.5, 'alpha_max_m2_s': 0.150022, 'alpha_mean_m2_s': 0.0837093458828125, 'stable_dt_limit_s': 9.998533548412901, 'central_building_temperature_C': 26.986545291531108, 'central_building_anomaly_C': 4.986545291531108, 'max_temp_C': 26.986545291531108, 'mean_temp_C': 22.171569732624352, 'max_anomaly_C': 4.986545291531108, 'mean_anomaly_C': 0.17156973262435127, 'total_anomaly_C_cells': 4392.185155183392, 'hot_area_gt_0p2C_cells': 2102, 'hot_area_gt_0p5C_cells': 1649, 'hot_area_gt_1C_cells': 1303, 'hot_area_gt_2C_cells': 873, 'hot_area_gt_5C_cells': 0, 'hot_area_gt_30C_cells': 0, 'hot_area_gt_35C_cells': 0, 'hot_area_gt_40C_cells': 0, 'hot_area_gt_45C_cells': 0, 'thermal_risk_objective': 26.986545291531108, 'building_cells': 14117, 'central_building_cells': 170, 'mask_source_area_m2': 1062.5, 'physical_roof_area_m2': 2531.0, 'source_area_m2': 2531.0, 'usable_power_density_W_m2': 987.7518767285658, 'source_heat_density_W_m2': 46085.05020900367, 'effective_source_heat_density_W_m2': 46085.05020900367, 'central_solar_heat_density_W_m2': 3.96, 'window_to_wall_ratio': 0.08, 'glass_area_m2': 266.24, 'opaque_wall_area_m2': 3061.76, 'total_wall_area_m2': 3328.0, 'UA_total_W_K': 2376.1920000000005, 'H_envelope_areal_W_m2K': 0.9388352429869619, 'heat_rejection_fraction': 1.0, 'ventilation_Vdot_m3_s': 5.624444444444444, 'dc_indoor_setpoint_C': 24.0, 'Q_it_MW': 87.39999999999999, 'COP': 4.263152173913043, 'PUE': 1.3345682160067311, 'Q_cooling_electric_MW': 20.501262078988297, 'Q_facility_MW': 116.64126207898829, 'Q_rejected_to_outdoor_MW': 116.64126207898829, 'Q_solar_opaque_MW': 0.91116, 'Q_solar_window_MW': 0.0559104, 'Q_solar_gain_MW': 0.9670704, 'Q_conduction_loss_MW': 0.006524084000000001, 'Q_ventilation_loss_MW': 0.013549432658521675, 'Q_convection_loss_MW': 0.6106179292383879, 'Q_radiation_loss_MW': 0.5394802374857848, 'Q_passive_net_MW': -0.2031012833826944, 'T_sky_C': 8.698667035086828, 'h_external_W_m2K': 20.9}, 'output_dir': 'load_optimization_outputs/opt_37fe16bf751a47/agent_round_0/02_TR1_Toronto_IBX_Data_Center_load_87.400MW'}],
                [68.0, 60.0, 55.0],
                [68.0, 63.6, 87.39999999999999],
                use_llm=True,
                thresholds=DatacenterThresholds(max_delta_t_with_concerns_c=9.5, plume_too_local_hot2_cells=400.0, plume_too_local_central_c=3.0, large_hot_footprint_hot5_cells=100.0, large_hot_footprint_central_c=10.0, downwind_max_anomaly_c=8.0, downwind_min_extra_mw=0.5),
                agent_round=0,
            )

In [22]:
reviews # reasoning behind each accept or reject verdict

[SiteReview(site_index=0, site_name='Toronto Edge DC', load_mw=68.0, verdict='accept', proposed_max_load_mw=None, reasons=['Toronto Edge DC accepted — plume OK (central ΔT 0.00 °C).'], concerns=[], source='rules', central_delta_t_c=0.0),
 SiteReview(site_index=1, site_name='Reykjavik Cold DC', load_mw=63.6, verdict='reject', proposed_max_load_mw=62.7, reasons=['Reykjavik Cold DC: downwind spread risk — reduce load (cap → 62.70 MW, central ΔT 0.00 °C).'], concerns=['downwind_spread_risk'], source='rules', central_delta_t_c=0.0),
 SiteReview(site_index=2, site_name='TR1 Toronto IBX Data Center', load_mw=87.39999999999999, verdict='accept', proposed_max_load_mw=None, reasons=['TR1 Toronto IBX Data Center accepted — plume OK (central ΔT 0.00 °C).'], concerns=[], source='rules', central_delta_t_c=0.0)]

In [23]:
all_ok # False means another iteration is requested; True means every agent accepted the workload

False